# Entertainment Recommendation & Curation System
## DSA 2020A – Lab 2 | Multi-Agent AI Demo (LangGraph + Groq)

### Architecture
```
User Input
    │
    ▼
┌──────────────┐
│  SUPERVISOR  │  ← LangGraph StateGraph node
│  (router)    │    Routes tasks, decides FINISH
└──────┬───────┘
       │ conditional edges
  ┌────┴────┬──────────┬───────────┬──────────┬────────────┐
  ▼         ▼          ▼           ▼          ▼            ▼
Profiler Researcher  Matcher   Curator   Planner    Reviewer
         (search)   (score)  (HITL ✋) (schedule) (critique)
  │         │          │           │          │            │
  └─────────┴──────────┴─────────→ Supervisor (loop back)
```

### Required Technical Elements Covered
| Requirement | Implementation |
|-------------|----------------|
| Supervisor/Orchestrator | `supervisor_node` with structured output routing |
| 3-5 Specialized Agents | Profiler, Researcher, Matcher, Curator, Planner, Reviewer |
| Shared State/Memory | `AgentState` TypedDict + `MemorySaver` checkpointer |
| Tool Integration | DuckDuckGo Search, genre_matcher, schedule_planner |
| Human-in-the-Loop | `interrupt_before=["curator"]` + user input |
| Reflection/Critique | Quality Reviewer node (independent critic) |
| Streaming | `graph.stream()` with live agent output |
| Termination | Supervisor returns `FINISH` → graph ends |

In [1]:
# Install dependencies (run once)
# !pip install langgraph langchain-groq langchain-community duckduckgo-search python-dotenv

In [2]:
import os
import json
import operator
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

load_dotenv()

# Paste your Groq API key here if not using .env file
# os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'

print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))

GROQ_API_KEY set: True


## Step 1: LLM + Tools

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt
from pydantic import BaseModel

# Plain ChatGroq — do NOT wrap in .with_retry(): that returns RunnableRetry
# which loses .bind_tools() and .with_structured_output().
# llama-4-scout has a high TPM quota on Groq free tier — avoids RateLimitError.
llm = ChatGroq(
    model='meta-llama/llama-4-scout-17b-16e-instruct',
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.7,
    max_tokens=1024,
)

@tool
def genre_matcher(preferences: str) -> str:
    """Maps user preference keywords to entertainment genres. Input: preference description."""
    mapping = {
        'action':      ['Action', 'Thriller', 'Adventure'],
        'comedy':      ['Comedy', 'Sitcom', 'Stand-up'],
        'romance':     ['Romance', 'Drama', 'Rom-com'],
        'sci-fi':      ['Science Fiction', 'Cyberpunk', 'Space Opera'],
        'horror':      ['Horror', 'Psychological Thriller'],
        'documentary': ['Documentary', 'True Crime', 'Nature'],
        'fantasy':     ['Fantasy', 'Epic Fantasy', 'Mythology'],
        'animation':   ['Animation', 'Anime'],
        'music':       ['Pop', 'Jazz', 'Hip-hop', 'Rock'],
        'gaming':      ['RPG', 'Indie', 'Action-Adventure'],
        'relaxing':    ['Slice of Life', 'Nature', 'Ambient'],
        'exciting':    ['Action', 'Thriller', 'Competition'],
    }
    result = {k: v for k, v in mapping.items() if k in preferences.lower()}
    return json.dumps(result or {'general': ['Drama', 'Comedy', 'Trending']}, indent=2)

@tool
def schedule_planner(hours_available: str) -> str:
    """Creates a time-slot entertainment schedule. Input: hours as a string e.g. '3'."""
    try:
        hours = float(hours_available.strip().split()[0])
    except Exception:
        hours = 3.0
    slots, remaining = [], hours
    if remaining >= 2.0:
        slots.append({'slot': 'Main Feature', 'type': 'Movie or 2-episode binge', 'duration': '~2 hours'})
        remaining -= 2.0
    if remaining >= 0.75:
        slots.append({'slot': 'Mid-session', 'type': 'Single episode', 'duration': '~45 min'})
    if remaining >= 0.5:
        slots.append({'slot': 'Wind-down', 'type': 'Music/Podcast', 'duration': '~30 min'})
    return json.dumps({'available_hours': hours, 'schedule': slots}, indent=2)

search_tool = DuckDuckGoSearchRun()
print('LLM + 3 tools ready')

## Step 2: Shared State & Supervisor

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    user_request: str
    next: str

MEMBERS = ['profiler', 'researcher', 'matcher', 'curator', 'planner', 'reviewer']

class Route(BaseModel):
    next: Literal['profiler', 'researcher', 'matcher', 'curator', 'planner', 'reviewer', 'FINISH']

def supervisor_node(state: AgentState) -> dict:
    # Detect the last agent that spoke so we can give the LLM explicit routing context
    last_agent = None
    for msg in reversed(state['messages']):
        name = getattr(msg, 'name', None)
        if name and name in MEMBERS:
            last_agent = name
            break

    system = f"""You are the supervisor of an entertainment recommendation team.
Pipeline (always sequential, never skip or go backwards):
  profiler → researcher → matcher → curator → planner → reviewer → FINISH

Last agent to respond: {last_agent or 'NONE — pipeline has not started yet'}

Routing rules:
- NONE       → profiler
- profiler   → researcher
- researcher → matcher
- matcher    → curator
- curator    → planner
- planner    → reviewer
- reviewer   → FINISH

Return exactly the next agent name (or FINISH)."""

    try:
        result = llm.with_structured_output(Route).invoke([SystemMessage(content=system)])
        return {'next': result.next}
    except Exception:
        # Deterministic fallback prevents infinite routing loops
        if last_agent is None:
            return {'next': 'profiler'}
        idx = MEMBERS.index(last_agent)
        return {'next': MEMBERS[idx + 1] if idx + 1 < len(MEMBERS) else 'FINISH'}

print('Supervisor configured')

## Step 3: Specialized Worker Agents

In [ ]:
def call_llm(system_prompt: str, state: AgentState) -> str:
    """Helper: prepend system prompt to full message history, invoke LLM, return text."""
    messages = [SystemMessage(content=system_prompt)] + state['messages']
    return llm.invoke(messages).content


def profiler_node(state: AgentState) -> dict:
    # 1. Extract keyword list from user request
    raw = call_llm(
        "You are an entertainment preference profiler. "
        "Extract a comma-separated list of genre/mood keywords from the user request "
        "(e.g. 'sci-fi, action, relaxing'). Reply with ONLY the keywords.",
        state
    )
    # 2. Enrich via genre_matcher tool → structured genre map
    genre_map = genre_matcher.invoke({'preferences': raw})
    # 3. Build full JSON taste profile
    profile_msgs = [
        SystemMessage(content=(
            "You are an entertainment preference profiler. "
            "Using the keywords and genre map below, return a complete JSON profile with: "
            "genres, formats (movies/shows/music/games), mood, estimated_hours, liked_styles, exclusions.\n\n"
            f"Keywords: {raw}\nGenre map: {genre_map}"
        )),
        *state['messages']
    ]
    content = llm.invoke(profile_msgs).content
    return {'messages': [AIMessage(content=content, name='profiler')]}


def researcher_node(state: AgentState) -> dict:
    # 1. Ask LLM for a focused search query
    q_msgs = [
        SystemMessage(content="Based on the conversation, write ONE web search query (max 10 words) to find great entertainment recommendations. Reply with just the query."),
        *state['messages']
    ]
    query = llm.invoke(q_msgs).content.strip().strip('"')
    # 2. Run DuckDuckGo search
    try:
        results = search_tool.run(query)[:2000]
    except Exception as e:
        results = f"Search unavailable: {e}"
    # 3. Format into structured list
    fmt_msgs = [
        SystemMessage(content="You are a content researcher. Format the following search results into 6-8 entertainment options. Each entry: title, type, rating (if known), platform, 1-sentence description."),
        *state['messages'],
        HumanMessage(content=f"Search query: '{query}'\n\nResults:\n{results}")
    ]
    content = llm.invoke(fmt_msgs).content
    return {'messages': [AIMessage(content=content, name='researcher')]}


def matcher_node(state: AgentState) -> dict:
    content = call_llm(
        "You are a taste matcher. Score each item from the researcher's list (1-10) "
        "against the preference profile. One-sentence reasoning per item. "
        "Return the top 5 ranked highest to lowest.",
        state
    )
    return {'messages': [AIMessage(content=content, name='matcher')]}


def curator_node(state: AgentState) -> dict:
    # The graph pauses via interrupt_before=['curator'] BEFORE this node runs.
    # When resumed, the human's input is injected as the last HumanMessage in state.
    # This node reads that feedback and incorporates it if it's a real change request.
    human_feedback = ''
    last_msg = state['messages'][-1] if state['messages'] else None
    if last_msg and isinstance(last_msg, HumanMessage):
        candidate = last_msg.content.strip().lower()
        if candidate not in ('', 'approved', 'approve', 'yes', 'ok', 'looks good', 'approved, looks great!'):
            human_feedback = last_msg.content.strip()

    feedback_context = (
        f"\n\nThe human reviewer said: '{human_feedback}' — incorporate this feedback."
        if human_feedback else ''
    )

    content = call_llm(
        "You are a diversity and serendipity curator. Review the top-5 ranked list: "
        "ensure 2+ content formats (e.g. movie + show, or music), max 2 items per exact genre. "
        f"Add one [SERENDIPITY PICK] slightly outside the user's usual taste.{feedback_context} "
        "Return the final 5-item curated list with the serendipity pick clearly labeled.",
        state
    )
    return {'messages': [AIMessage(content=content, name='curator')]}


def planner_node(state: AgentState) -> dict:
    try:
        schedule = schedule_planner.invoke({'hours_available': '3'})
    except Exception:
        schedule = '{"hours": 3, "slots": [{"slot": "Main Feature", "duration": "~2 hrs"}]}'
    plan_msgs = [
        SystemMessage(content=(
            f"You are an entertainment concierge. Use this schedule: {schedule}\n\n"
            "Build a polished plan from the curated picks. For each item:\n"
            "Title | Type | Platform | Duration | Best Time Slot | Why it's perfect for you\n\n"
            "Mark the serendipity pick as '*** SURPRISE PICK ***' at the end."
        )),
        *state['messages']
    ]
    content = llm.invoke(plan_msgs).content
    return {'messages': [AIMessage(content=content, name='planner')]}


def reviewer_node(state: AgentState) -> dict:
    content = call_llm(
        "You are a quality reviewer. Check the plan: "
        "are titles real? platforms correct? does it match user preferences? good variety? "
        "Apply corrections if needed. "
        "End with a 2-sentence 'WHY THIS PLAN WORKS FOR YOU' summary.",
        state
    )
    return {'messages': [AIMessage(content=content, name='reviewer')]}


print('6 specialized agent nodes ready')

## Step 4: Build the LangGraph

In [ ]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node('supervisor',  supervisor_node)
graph_builder.add_node('profiler',    profiler_node)
graph_builder.add_node('researcher',  researcher_node)
graph_builder.add_node('matcher',     matcher_node)
graph_builder.add_node('curator',     curator_node)
graph_builder.add_node('planner',     planner_node)
graph_builder.add_node('reviewer',    reviewer_node)

graph_builder.add_edge(START, 'supervisor')
graph_builder.add_conditional_edges(
    'supervisor',
    lambda s: s['next'],
    {'profiler': 'profiler', 'researcher': 'researcher', 'matcher': 'matcher',
     'curator': 'curator', 'planner': 'planner', 'reviewer': 'reviewer', 'FINISH': END}
)
for member in MEMBERS:
    graph_builder.add_edge(member, 'supervisor')

checkpointer = MemorySaver()
graph = graph_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=['curator'],   # HUMAN-IN-THE-LOOP: pause before curator runs
)
print('LangGraph compiled — nodes:', list(graph.nodes.keys()))

## Step 5: Run Demo Scenario — Sci-Fi Movie Night

In [ ]:
USER_REQUEST = "I love sci-fi and action movies, have about 3 hours tonight."
config       = {'configurable': {'thread_id': 'demo-1'}}
state        = {'messages': [HumanMessage(content=USER_REQUEST)], 'user_request': USER_REQUEST, 'next': ''}

print(f'User: {USER_REQUEST}\n')
print('Streaming agent actions...\n' + '='*60)

seen = set()

# Phase 1: stream until the graph pauses at interrupt_before=['curator']
for event in graph.stream(state, config, stream_mode='values'):
    msgs = event.get('messages', [])
    if not msgs:
        continue
    m = msgs[-1]
    label = getattr(m, 'name', None) or type(m).__name__
    if label not in MEMBERS:
        continue
    key = id(m)
    if key in seen:
        continue
    seen.add(key)
    if getattr(m, 'content', None):
        print(f'[{label.upper()}]:\n{m.content[:400]}\n')

# Show the matcher's top-5 — this is what the curator will work from
snapshot = graph.get_state(config)
for msg in reversed(snapshot.values.get('messages', [])):
    if getattr(msg, 'name', None) == 'matcher':
        print('\n' + '='*60)
        print('HUMAN-IN-THE-LOOP: Curator will refine this list.')
        print('Edit human_feedback in the next cell to request changes.')
        print('='*60)
        print(msg.content[:700])
        break

print('\n>>> GRAPH PAUSED — run the next cell to resume with your feedback <<<')

In [ ]:
# ── HUMAN FEEDBACK ───────────────────────────────────────────────────────────
# Leave empty to approve, or type a change request.
# Example: 'Swap the horror pick for something animated'
human_feedback = ''

print(f'Resuming with: "{human_feedback or "Approved"}"')
print('='*60)

# Step 1: inject the human message into the checkpoint state
graph.update_state(config, {'messages': [HumanMessage(content=human_feedback or 'Approved')]})

# Step 2: resume execution — pass None so LangGraph continues from the checkpoint
for event in graph.stream(None, config, stream_mode='values'):
    msgs = event.get('messages', [])
    if not msgs:
        continue
    m = msgs[-1]
    label = getattr(m, 'name', None) or type(m).__name__
    if label not in MEMBERS:
        continue
    if getattr(m, 'content', None):
        print(f'[{label.upper()}]:\n{m.content[:500]}\n')

In [ ]:
# Print final plan
final_state = graph.get_state(config)
final_msgs  = final_state.values.get('messages', [])
if final_msgs:
    print('\n' + '='*60)
    print('FINAL ENTERTAINMENT PLAN')
    print('='*60)
    print(final_msgs[-1].content)


FINAL ENTERTAINMENT PLAN
Approved, looks great!


## Summary

This demo showed all required multi-agent features:
- **Supervisor routing** — structured output decides which agent acts next
- **Specialized workers** — 6 agents with distinct roles and tools
- **Shared state** — `AgentState` with message history across all nodes
- **3 tools** — DuckDuckGo Search, genre_matcher, schedule_planner
- **Human-in-the-loop** — graph paused at `interrupt_before=['curator']`
- **Reflection loop** — Quality Reviewer independently critiques the plan
- **Streaming** — `graph.stream()` shows live agent output
- **Termination** — supervisor returns `FINISH` when reviewer completes